# 04: Modeling & Risk Scoring (Phase 4/5)

Original 6-fund modeling work, split out from the combined EDA+modeling
notebook. Continues directly from `03_eda_feature_engineering.ipynb`'s
`long_df` / `long_df_encoded` / `x_train`/`x_test` etc. -- run `03` first,
or re-load equivalent variables, before running this notebook.

## What's here
- Phase 4a/4b: Linear Regression + Random Forest baselines predicting mean
  CAGR (informative negative result -- R^2 ~0.17, documented below)
- Phase 4c/4d: risk-adjusted scoring (`score = mean - penalty * max(-min, 0)`),
  `recommend_fund()` / `advise_investment()`
- Phase 5: quantile regression (GradientBoostingRegressor) as an ML-based
  alternative to the hand-computed historical `min_cagr` risk measure

This is the original 6-fund version. The 138-fund expansion (same ideas,
category-based encoding instead of per-fund) lives in
`05_expanded_dataset_pipeline.ipynb`.

In [1]:
import pandas as pd
import numpy as np

**Self-contained setup** -- rebuilds `model_df` / `long_df_encoded` /
`x_train` etc. from scratch rather than depending on `03`'s in-memory
state. In Jupyter/VS Code, each notebook file runs in its own separate
kernel by default, so variables from `03` are *not* automatically
available here even if you ran `03` first in another tab. This section
recomputes everything needed (fast -- only 6 funds, not the 138-fund
pipeline in `05`).

In [2]:
fund_names = ['hdfc_large_cap', 'hdfc_corporate_bond', 'hdfc_hybrid_equity',
              'sbi_large_cap', 'icici_large_cap', 'icici_corporate_bond']

data = {}
for name in fund_names:
    df = pd.read_csv(f"../Data/raw/{name}.csv")
    df['date'] = pd.to_datetime(df['date'])
    data[name] = df

print(data.keys())

dict_keys(['hdfc_large_cap', 'hdfc_corporate_bond', 'hdfc_hybrid_equity', 'sbi_large_cap', 'icici_large_cap', 'icici_corporate_bond'])


In [3]:
def calculate_cagr(df):
    nav_start = df['nav'].iloc[0]
    nav_end = df['nav'].iloc[-1]
    date_start = df['date'].iloc[0]
    date_end = df['date'].iloc[-1]
    years = (date_end - date_start).days / 365.25
    return (nav_end / nav_start) ** (1 / years) - 1


def calculate_rolling_cagr(df, window_years):
    df = df.copy()
    df = df.set_index('date')

    rolling_cagr = []
    dates = []

    for i in range(len(df)):
        start_date = df.index[i]
        target_end_date = start_date + pd.DateOffset(years=window_years)

        if target_end_date > df.index[-1]:
            break
        end_idx = df.index.searchsorted(target_end_date)

        start_nav = df['nav'].iloc[i]
        end_nav = df['nav'].iloc[end_idx]

        actual_years = (df.index[end_idx] - start_date).days / 365.25
        cagr = (end_nav / start_nav) ** (1 / actual_years) - 1
        rolling_cagr.append(cagr)
        dates.append(start_date)
    return pd.Series(rolling_cagr, index=dates)

In [4]:
window_list = [x for x in range(1, 11)]
long_data = []

for name, df in data.items():
    for window in window_list:
        rolling = calculate_rolling_cagr(df, window)
        for date, rolling_cagr in rolling.items():
            long_data.append({
                'fund': name,
                'window_years': window,
                'start_date': date,
                'cagr': rolling_cagr
            })

long_df = pd.DataFrame(long_data)
print(long_df.shape)

(117804, 4)


In [5]:
long_df_encoded = pd.get_dummies(long_df, columns=['fund'])
model_df = long_df_encoded.drop(columns=['start_date'])

print(model_df.shape)
print(model_df.columns.tolist())

(117804, 8)
['window_years', 'cagr', 'fund_hdfc_corporate_bond', 'fund_hdfc_hybrid_equity', 'fund_hdfc_large_cap', 'fund_icici_corporate_bond', 'fund_icici_large_cap', 'fund_sbi_large_cap']


## Original Phase 4/5 content continues below

<!-- TODO: SEPERATING INPUT X AND TARGET(PREDICTION) Y -->

In [6]:
x= model_df.drop(columns =['cagr'])

y = model_df['cagr']

# quantile is take x% of y for datetime type objext

cutoff_date = long_df_encoded['start_date'].quantile(.8)

print(x.shape)

print(y.shape)

print(cutoff_date)

(117804, 7)
(117804,)
2020-05-20 00:00:00


In [7]:
train_dates = long_df_encoded['start_date']< cutoff_date 

test_dates = long_df_encoded['start_date'] >= cutoff_date



x_train, x_test = x[train_dates] , x[test_dates]

y_train, y_test = y[train_dates], y[test_dates]



print(x_train.shape, x_test.shape)

print(y_train.shape, y_test.shape)

(94230, 7) (23574, 7)
(94230,) (23574,)


In [8]:
!pip install scikit-learn


In [9]:
from sklearn.linear_model import LinearRegression



model = LinearRegression()

model.fit(x_train, y_train)


,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](7,)","[-0. ,-0.04, 0. ,...,-0.04, 0.03, 0.03]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](7,)","['window_years','fund_hdfc_corporate_bond','fund_hdfc_hybrid_equity',..., 'fund_icici_corporate_bond','fund_icici_large_cap','fund_sbi_large_cap']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,0.124
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,7
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(6)


In [10]:
print(x.columns.tolist)

print(model.coef_)



print(model.intercept_)

<bound method IndexOpsMixin.tolist of Index(['window_years', 'fund_hdfc_corporate_bond', 'fund_hdfc_hybrid_equity',
       'fund_hdfc_large_cap', 'fund_icici_corporate_bond',
       'fund_icici_large_cap', 'fund_sbi_large_cap'],
      dtype='str')>
[-0.00077221 -0.03659973  0.00304254  0.014573   -0.03736511  0.02870923
  0.02764006]
0.12399917431540096


In [11]:
y_pred = model.predict(x_test)


In [12]:
from sklearn.metrics import mean_absolute_error, r2_score

mae = mean_absolute_error(y_test, y_pred)

r2 = r2_score(y_test, y_pred)



print('Mae: ', mae)

print('r2: ', r2)

Mae:  0.056740769744004306
r2:  0.17413878955319595


Phase 4a: Baseline Linear Regression — Results & Analysis

Setup: Trained on long_df (117,804 rolling-window CAGR rows across 6 funds, window_years 1–10), one-hot encoded by fund, split 80/20 by start_date (time-based split, not random, to avoid leaking near-duplicate overlapping windows between train and test).

Results:



MAE: 0.0567 (predictions off by ~5.7 percentage points of CAGR on average)

R²: 0.174 (model explains ~17% of the variation in actual CAGR)



Why it's okay as a baseline:



It correctly ranks the funds in a way that's consistent with Phase 3's EDA — icici_large_cap and sbi_large_cap come out with the highest coefficients, hdfc_corporate_bond and icici_corporate_bond the lowest, matching the equity-vs-bond return pattern already observed.

It confirms the pipeline (data → features → split → train → evaluate) works end-to-end before adding complexity — useful as a reference point to measure future models against.



Why it fails / known limitations:



window_years coefficient is near-zero (-0.00077). The model has effectively learned that holding period barely affects predicted CAGR at all. This contradicts Phase 3's finding of a "dip then recovery" pattern in CAGR across window lengths, and the finding that equity funds show sharply reduced downside risk at longer horizons.

Root cause: Linear Regression cannot represent interaction effects. It assumes each feature (fund, window_years) affects CAGR independently and additively. In reality, the effect of holding period likely depends on which fund it is — e.g. an equity fund may need years to recover from short-term dips while a bond fund's return stays flat regardless of horizon. Since the model has no way to represent "fund A behaves differently over time than fund B," it averages this variation away, which shows up as the flat window_years coefficient.

Practical consequence: because holding period barely moves the prediction, the model's fund ranking is nearly static — it would recommend the same 1–2 funds (icici_large_cap, sbi_large_cap) almost regardless of the number of years input by the user. This defeats the actual purpose of an investment advisor tool, which should ideally recommend different funds for different time horizons and risk profiles.

Dummy variable trap: all 6 fund columns were kept during one-hot encoding (none dropped as baseline via drop_first=True), so individual fund coefficients and the intercept are not uniquely identifiable — only their relative ordering is meaningful. Noted as a data-prep fix for future iterations, though it doesn't affect predictions themselves.



Conclusion: Linear Regression is too structurally simple for this problem — the real relationship between fund, holding period, and return appears non-linear, with time-dependent behavior that differs per fund. This motivates trying a model that can learn interactions and non-linear patterns natively, e.g. a tree-based model (Random Forest), rather than manually engineering interaction terms into a linear model.

In [13]:
from sklearn.ensemble import RandomForestRegressor

# estimators is no of dictinct trees to use and ranfom state fixes the output like seed in random.integer

rf_model = RandomForestRegressor(n_estimators = 100, random_state = 42)

rf_model.fit(x_train, y_train)






,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease o

In [14]:
y_pred_rf = rf_model.predict(x_test)

mae_rf = mean_absolute_error(y_test,y_pred_rf)

r2_rf = r2_score(y_test, y_pred_rf)



print('mae: ', mae_rf)

print('r2: ', r2_rf)

mae:  0.05887305311389572
r2:  0.1636818497235215


Phase 4b: Random Forest — Results & Analysis

Setup: Same features (window_years + 6 fund dummies), same time-based 80/20 train/test split as the Linear Regression baseline, for direct comparison. RandomForestRegressor(n_estimators=100, random_state=42).

Results:

MetricLinear RegressionRandom ForestMAE0.05670.0589R²0.1740.164

Random Forest performed slightly worse than the linear baseline on both metrics, despite being a more flexible, non-linear model.

Why the more flexible model didn't help:



Feature space is too small to exploit. With only window_years (10 possible values) and 6 one-hot fund columns, there are just 60 unique (fund, window_years) combinations across all 117,804 rows. Both models are fundamentally limited to learning something close to the average CAGR per bucket — there isn't enough distinct feature information for Random Forest's added flexibility to meaningfully outperform a straight line. The "extra power" had little real signal left to extract.

Likely mild overfitting. With 100 unrestricted trees, Random Forest can fit noise within each bucket's training rows (variation driven by which historical period a window happened to start in — information the model never actually sees, since start_date isn't a feature). This noise doesn't generalize to the test set, and can slightly hurt test performance relative to a simpler model that isn't flexible enough to overfit in the first place.

Train/test split spans a market regime shift. The time-based split trains on ~2013–2020 and tests on ~2020–2026 — a period that includes the COVID crash and recovery. A model trained only on pre-2020 data has no way to anticipate this kind of shock. This affects both models roughly equally and is a genuine limitation of the ~13.5-year dataset (already flagged in Phase 3), not a modeling error.



Conclusion: The bottleneck is not model choice — it's feature information. window_years and fund alone don't carry enough signal to predict CAGR well, regardless of how flexible the model is. This confirms that the previously-logged future extension, "market conditions as a feature" (e.g. macro indicators, rolling market volatility, or time-since-start as a feature), is not just a nice-to-have but likely the actual next lever for improving prediction quality. For the current scope, Linear Regression's marginal edge and interpretability make it the more defensible baseline model to carry forward, with this comparison documented as evidence that added model complexity alone does not solve the problem.

A SOLN TO THIS COULD BE TO ADD ANOTHER COLUMN TO PROVIDE EXTRA DATA TO THE PREDICTIONS BUT IF U THINK ABOUT IT STILL WONT GET DIFF RESULT. FOR EX THE ICICI LARGE CAP STAYS ON TOP FOR ALL THE YEARS SO EVEN IF THE PREDICITONS GET CLOSER TO ACCTUAL VALUES IT WONT CANGE THE FUND, IT MIGHT CHANGE THE AMMOUNT BUT THE FUND STAYS THE SAME SO THE ANS STAGNATES. SO TO SOLVE THIS WE'LL ADD ANOTHER PAREAMETER BASED ON THE MIN AND STD DEV AS YEAR BY YEAR THE FALL AND RISE OF A FUND IS NOT FIXED SO WE'LL GET A MUCH MORE ACCURATE AND VARIED ANS


RISK CALCULATION

In [15]:
risk_return_df = long_df.groupby(['fund', 'window_years'])['cagr'].agg(mean = 'mean', min = 'min').reset_index()

print(risk_return_df)

                    fund  window_years      mean       min
0    hdfc_corporate_bond             1  0.080809  0.017989
1    hdfc_corporate_bond             2  0.080870  0.038043
2    hdfc_corporate_bond             3  0.079915  0.050127
3    hdfc_corporate_bond             4  0.079049  0.057946
4    hdfc_corporate_bond             5  0.078948  0.060465
5    hdfc_corporate_bond             6  0.080203  0.062887
6    hdfc_corporate_bond             7  0.080533  0.070348
7    hdfc_corporate_bond             8  0.079513  0.072398
8    hdfc_corporate_bond             9  0.078806  0.071786
9    hdfc_corporate_bond            10  0.079527  0.074101
10    hdfc_hybrid_equity             1  0.151122 -0.262422
11    hdfc_hybrid_equity             2  0.133907 -0.130787
12    hdfc_hybrid_equity             3  0.126412 -0.059587
13    hdfc_hybrid_equity             4  0.126410  0.015157
14    hdfc_hybrid_equity             5  0.123799 -0.011455
15    hdfc_hybrid_equity             6  0.120533  0.0354

In [16]:
penalty_weight = 0.5





risk_return_df['score'] = risk_return_df['mean'] - penalty_weight * risk_return_df['min'].apply(lambda x: max(-x, 0))



risk_return_df.loc[risk_return_df.groupby('window_years')['score'].idxmax(), ['fund' , 'window_years', 'score']]

,fund,window_years,score
30,icici_corporate_bond,1,0.081429
41,icici_large_cap,2,0.094806
42,icici_large_cap,3,0.131941
43,icici_large_cap,4,0.156395
44,icici_large_cap,5,0.154617
45,icici_large_cap,6,0.150432
46,icici_large_cap,7,0.147071
47,icici_large_cap,8,0.152449
48,icici_large_cap,9,0.154203
49,icici_large_cap,10,0.155933


In [17]:
for weight in [.2,.5, 1, 1.5, 2]:

    risk_return_df['score'] = risk_return_df['mean'] - (weight * risk_return_df['min']).apply(lambda x: max(-x, 0))



    print(f" penalty weight: ", weight)

    print(risk_return_df.loc[risk_return_df.groupby('window_years')['score'].idxmax()],)

    

 penalty weight:  0.2
               fund  window_years      mean       min     score
40  icici_large_cap             1  0.174532 -0.315764  0.111379
41  icici_large_cap             2  0.163715 -0.137818  0.136151
42  icici_large_cap             3  0.157875 -0.051867  0.147501
43  icici_large_cap             4  0.156395  0.017597  0.156395
44  icici_large_cap             5  0.154617  0.000133  0.154617
45  icici_large_cap             6  0.150432  0.060959  0.150432
46  icici_large_cap             7  0.147071  0.079356  0.147071
47  icici_large_cap             8  0.152449  0.110737  0.152449
48  icici_large_cap             9  0.154203  0.127106  0.154203
49  icici_large_cap            10  0.155933  0.128354  0.155933
 penalty weight:  0.5
                    fund  window_years      mean       min     score
30  icici_corporate_bond             1  0.081429  0.030581  0.081429
41       icici_large_cap             2  0.163715 -0.137818  0.094806
42       icici_large_cap             3  0.157

ANALYSYS : The reason equity dominates so heavily after the crossover point isn't a flaw — it's because your data genuinely shows equity's downside risk collapses fast (by window_years=4, min_cagr is already positive, 0.0176), so once the "danger zone" passes, there's no more reason for the penalty to hold it back.

In [18]:
import importlib
import sys
sys.path.append('../src')

import calculators

importlib.reload(calculators)

from calculators import recommend_fund, advise_investment , advise_investment_web



advise_investment(risk_return_df, 100000, 3,2)

Top 1 fund(s) recommened for 3 years at penalty weight 2 (risk measure: min):

  #1: icici_corporate_bond
      Expected Cagr=0.0800 
      Worst Case Cagr return (min)=0.0553
      Score= 0.0800


Projected value of Rs100000 over 3 years, per recommended fund:
  icici_corporate_bond:
      Balanced estimate:      approx Rs121741.85
      Best case (mean):       approx Rs125965.54
      Worst case (min):   approx Rs117518.15


,fund,mean,min,score
0,icici_corporate_bond,0.079984,0.055281,0.079984


In [19]:
recommend_fund(risk_return_df, 5, 0.5)

recommend_fund(risk_return_df, 1, 1.0)

recommend_fund(risk_return_df, 3, 3.5)

Top 1 fund(s) recommened for 5 years at penalty weight 0.5 (risk measure: min):

  #1: icici_large_cap
      Expected Cagr=0.1546 
      Worst Case Cagr return (min)=0.0001
      Score= 0.1546

Top 1 fund(s) recommened for 1 years at penalty weight 1.0 (risk measure: min):

  #1: icici_corporate_bond
      Expected Cagr=0.0814 
      Worst Case Cagr return (min)=0.0306
      Score= 0.0814

Please enter a valid penalty range


In [20]:
advise_investment(risk_return_df, 10000, 2, 2)

Top 1 fund(s) recommened for 2 years at penalty weight 2 (risk measure: min):

  #1: icici_corporate_bond
      Expected Cagr=0.0809 
      Worst Case Cagr return (min)=0.0433
      Score= 0.0809


Projected value of Rs10000 over 2 years, per recommended fund:
  icici_corporate_bond:
      Balanced estimate:      approx Rs11284.33
      Best case (mean):       approx Rs11683.45
      Worst case (min):   approx Rs10885.20


,fund,mean,min,score
0,icici_corporate_bond,0.0809,0.043322,0.0809


In [21]:
from sklearn.ensemble import GradientBoostingRegressor

quantile_model = GradientBoostingRegressor(

    loss= 'quantile',

    alpha = .05,

    n_estimators= 100,

    random_state = 42

)

quantile_model.fit(x_train, y_train)

y_pred_quantile = quantile_model.predict(x_test)

print(y_pred_quantile[:10])

print(x_test.iloc[:10])

[-0.16200775 -0.16200775 -0.16200775 -0.16200775 -0.16200775 -0.16200775
 -0.16200775 -0.16200775 -0.16200775 -0.16200775]
      window_years  fund_hdfc_corporate_bond  fund_hdfc_hybrid_equity  \
1811             1                     False                    False   
1812             1                     False                    False   
1813             1                     False                    False   
1814             1                     False                    False   
1815             1                     False                    False   
1816             1                     False                    False   
1817             1                     False                    False   
1818             1                     False                    False   
1819             1                     False                    False   
1820             1                     False                    False   

      fund_hdfc_large_cap  fund_icici_corporate_bond  fund_icici_large_ca

In [22]:
print(np.unique(y_pred_quantile))

print(x_test.groupby(['window_years']).size())

[-1.62007747e-01 -1.19245237e-01 -9.69277626e-02 -7.68337149e-02
 -3.48973269e-02 -1.36862151e-02 -5.52438443e-03 -3.11192157e-03
 -9.31430253e-04 -7.84217789e-05  1.89814250e-02  2.79033098e-02
  3.88095874e-02  3.91325233e-02  4.57242055e-02  4.85127236e-02
  4.87385216e-02  5.26377383e-02  5.62666643e-02  5.80390124e-02
  5.94779863e-02  5.98009222e-02  6.00557288e-02  6.24585366e-02
  6.64457291e-02  6.66332676e-02  6.78651918e-02  6.82802019e-02
  7.04693309e-02  7.08951328e-02  7.09349673e-02  7.12579032e-02
  7.13627276e-02  7.20985988e-02  8.85304282e-02  9.12857966e-02]
window_years
1    7608
2    6125
3    4672
4    3205
5    1727
6     237
dtype: int64


In [23]:
funds = ['hdfc_corporate_bond', 'hdfc_hybrid_equity', 'hdfc_large_cap',

         'icici_corporate_bond', 'icici_large_cap', 'sbi_large_cap']



all_combos = []

for fund in funds:

    for years in range(1,11):

        row = {'window_years': years}

        for f in funds:

            row[f'fund_{f}'] = (f == fund)

        all_combos.append(row)





x_all = pd.DataFrame(all_combos)

x_all = x_all[x.columns]

print(x_all.shape)


(60, 7)


In [24]:
y_pred_all = quantile_model.predict(x_all)



comparison_df = x_all.copy()

comparison_df['predicted_p10_cagr'] = y_pred_all



comparison_df['fund'] = comparison_df[[f'fund_{f}' for f in funds]].idxmax(axis=1).str.replace('fund_', '')



final_comparison = comparison_df.merge(

    risk_return_df[['fund', 'window_years', 'min']],

    on=['fund', 'window_years']

)

final_comparison = final_comparison[['fund', 'window_years', 'predicted_p10_cagr', 'min']]

print(final_comparison.sort_values(['fund', 'window_years']))

                    fund  window_years  predicted_p10_cagr       min
0    hdfc_corporate_bond             1            0.045724  0.017989
1    hdfc_corporate_bond             2            0.060056  0.038043
2    hdfc_corporate_bond             3            0.066633  0.050127
3    hdfc_corporate_bond             4            0.067865  0.057946
4    hdfc_corporate_bond             5            0.068280  0.060465
5    hdfc_corporate_bond             6            0.071363  0.062887
6    hdfc_corporate_bond             7            0.072366  0.070348
7    hdfc_corporate_bond             8            0.074255  0.072398
8    hdfc_corporate_bond             9            0.073693  0.071786
9    hdfc_corporate_bond            10            0.075352  0.074101
10    hdfc_hybrid_equity             1           -0.119245 -0.262422
11    hdfc_hybrid_equity             2           -0.013686 -0.130787
12    hdfc_hybrid_equity             3           -0.000078 -0.059587
13    hdfc_hybrid_equity          

In [25]:
y_pred_final = quantile_model.predict(x_all)



ml_risk_df = x_all.copy()

ml_risk_df['fund'] = ml_risk_df[[f'fund_{f}' for f in funds]].idxmax(axis=1).str.replace('fund_', '')

ml_risk_df['predicted_min_cagr'] = y_pred_final

ml_risk_df = ml_risk_df.merge(risk_return_df[['fund', 'window_years', 'mean']], on=['fund', 'window_years'])

ml_risk_df = ml_risk_df[['fund', 'window_years', 'mean', 'predicted_min_cagr']]

print(ml_risk_df)

                    fund  window_years      mean  predicted_min_cagr
0    hdfc_corporate_bond             1  0.080809            0.045724
1    hdfc_corporate_bond             2  0.080870            0.060056
2    hdfc_corporate_bond             3  0.079915            0.066633
3    hdfc_corporate_bond             4  0.079049            0.067865
4    hdfc_corporate_bond             5  0.078948            0.068280
5    hdfc_corporate_bond             6  0.080203            0.071363
6    hdfc_corporate_bond             7  0.080533            0.072366
7    hdfc_corporate_bond             8  0.079513            0.074255
8    hdfc_corporate_bond             9  0.078806            0.073693
9    hdfc_corporate_bond            10  0.079527            0.075352
10    hdfc_hybrid_equity             1  0.151122           -0.119245
11    hdfc_hybrid_equity             2  0.133907           -0.013686
12    hdfc_hybrid_equity             3  0.126412           -0.000078
13    hdfc_hybrid_equity          

In [26]:
recommend_fund(risk_return_df, 3, 1.0)                                  # historical min

recommend_fund(ml_risk_df, 3, 1.0, risk_column='predicted_min_cagr')    # ML-predicted

Top 1 fund(s) recommened for 3 years at penalty weight 1.0 (risk measure: min):

  #1: icici_large_cap
      Expected Cagr=0.1579 
      Worst Case Cagr return (min)=-0.0519
      Score= 0.1060

Top 1 fund(s) recommened for 3 years at penalty weight 1.0 (risk measure: predicted_min_cagr):

  #1: icici_large_cap
      Expected Cagr=0.1579 
      Worst Case Cagr return (predicted_min_cagr)=0.0279
      Score= 0.1579



,fund,mean,predicted_min_cagr,score
0,icici_large_cap,0.157875,0.027903,0.157875


In [27]:
recommend_fund(risk_return_df, 1, 1.5)

recommend_fund(ml_risk_df, 1, 1.5, risk_column='predicted_min_cagr')

advise_investment(ml_risk_df, 10000,10,1.5, risk_column = 'predicted_min_cagr')

Top 1 fund(s) recommened for 1 years at penalty weight 1.5 (risk measure: min):

  #1: icici_corporate_bond
      Expected Cagr=0.0814 
      Worst Case Cagr return (min)=0.0306
      Score= 0.0814

Top 1 fund(s) recommened for 1 years at penalty weight 1.5 (risk measure: predicted_min_cagr):

  #1: icici_corporate_bond
      Expected Cagr=0.0814 
      Worst Case Cagr return (predicted_min_cagr)=0.0526
      Score= 0.0814

Top 1 fund(s) recommened for 10 years at penalty weight 1.5 (risk measure: predicted_min_cagr):

  #1: icici_large_cap
      Expected Cagr=0.1559 
      Worst Case Cagr return (predicted_min_cagr)=0.1353
      Score= 0.1559


Projected value of Rs10000 over 10 years, per recommended fund:
  icici_large_cap:
      Balanced estimate:      approx Rs39080.29
      Best case (mean):       approx Rs42591.84
      Worst case (predicted_min_cagr):   approx Rs35568.73


,fund,mean,predicted_min_cagr,score
0,icici_large_cap,0.155933,0.13529,0.155933


Phase 5: ML-Driven Risk Estimation — Quantile Regression

Motivation: the earlier recommend_fund() (Phase 4c) used a hand-computed historical minimum (min_cagr) as its risk measure — accurate, but entirely rule-based, with no actual model in the recommendation path. This section replaces that hand-computed number with a genuinely trained ML model, directly addressing the goal of having real prediction, not just statistics, driving the recommendation.

Why regression on mean CAGR (Phase 4a/4b) wasn't the right target for this: those models tried to predict average return, which barely varies with window_years — that's why Linear Regression and Random Forest both underperformed (R² ≈ 0.17). The real, learnable signal isn't in the mean — it's in how risk (not return) shrinks with holding period, which Phase 3's EDA and Phase 4c's min_cagr table both showed clearly.

Approach — Quantile Regression: trained GradientBoostingRegressor(loss='quantile', alpha=...) on the full row-level long_df (117,804 rows, same features as before: window_years + fund dummies). Unlike standard regression (which predicts the mean), quantile regression predicts a specific percentile of the outcome distribution — e.g. alpha=0.05 asks the model to learn "the CAGR this fund/horizon combination falls below only 5% of the time," directly analogous to (but statistically more robust than) a historical worst-case.

Alpha tuning — tested 0.1, 0.05, and 0.01:

alphaBehavior0.1Reasonable predictions, but noticeably less pessimistic than actual historical min at short horizons0.05Closer to historical min, while staying smooth and monotonic across window_years — best balance0.01Closer still to min on average, but noticeably noisy: non-monotonic zigzags across adjacent window_years for some funds, and at least one case where the prediction was more extreme than the fund's actual all-time worst case — a sign of overfitting to a thin, correlated sample rather than learning real structure

Settled on alpha=0.05 as the final model, balancing closeness to historical extremes against reliability/smoothness.

Integration: recommend_fund() and advise_investment() in src/calculators.py were extended with a risk_column parameter (default 'min', preserving backward compatibility), allowing either the original historical risk measure or the new ML-predicted risk measure (predicted_min_cagr, stored in ml_risk_df) to be used interchangeably in the same scoring formula.

Comparison — historical min vs. ML predicted_min_cagr: tested side-by-side across multiple (window_years, penalty) combinations. In every case tested, both risk measures agreed on which fund to recommend. Where they differed was in severity: the historical minimum, being a single extreme outlier, tends to apply a harsher risk penalty at short horizons than the smoother ML estimate — e.g. at window_years=3, penalty=1.0, the same fund (icici_large_cap) scored 0.106 under historical risk vs. 0.158 under ML-predicted risk, purely because the ML model's 5th-percentile estimate wasn't negative while the historical worst-case was.

Conclusion: the ML-based risk measure validates the original hand-built approach — it doesn't contradict which funds get recommended, but offers a more statistically grounded alternative to a single historical extreme, less sensitive to one-off outlier events. This directly answers the earlier concern about the recommendation engine relying entirely on hand-written statistics: the pipeline now has a genuine trained model — with tuned hyperparameters and validated predictions — sitting alongside (not replacing) the original approach, giving the tool two defensible, cross-validated risk perspectives instead of one.

In [28]:
recommend_fund(risk_return_df, 3, 2, n=3)

advise_investment(risk_return_df, 100, 1, 2, n=3)

Top 3 fund(s) recommened for 3 years at penalty weight 2 (risk measure: min):

  #1: icici_corporate_bond
      Expected Cagr=0.0800 
      Worst Case Cagr return (min)=0.0553
      Score= 0.0800

  #2: hdfc_corporate_bond
      Expected Cagr=0.0799 
      Worst Case Cagr return (min)=0.0501
      Score= 0.0799

  #3: icici_large_cap
      Expected Cagr=0.1579 
      Worst Case Cagr return (min)=-0.0519
      Score= 0.0541

Top 3 fund(s) recommened for 1 years at penalty weight 2 (risk measure: min):

  #1: icici_corporate_bond
      Expected Cagr=0.0814 
      Worst Case Cagr return (min)=0.0306
      Score= 0.0814

  #2: hdfc_corporate_bond
      Expected Cagr=0.0808 
      Worst Case Cagr return (min)=0.0180
      Score= 0.0808

  #3: hdfc_hybrid_equity
      Expected Cagr=0.1511 
      Worst Case Cagr return (min)=-0.2624
      Score= -0.3737


Projected value of Rs100 over 1 years, per recommended fund:
  icici_corporate_bond:
      Balanced estimate:      approx Rs105.60
      Be

,fund,mean,min,score
0,icici_corporate_bond,0.081429,0.030581,0.081429
1,hdfc_corporate_bond,0.080809,0.017989,0.080809
2,hdfc_hybrid_equity,0.151122,-0.262422,-0.373722
